<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/40_apply_rules_to_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# ==========================================
# 40_apply_rules_to_models.ipynb
# ==========================================
# Purpose:
#   - Take extracted model outputs (probabilities / positions)
#   - Attach missing Close prices from the engineered dataset
#   - Apply trading rules to generate signals
#   - Compute equity curve and performance metrics
#   - Save final artefacts for the demo app
# ==========================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, shutil, math, os

# -----------------------
# Project paths
# -----------------------
PROJECT_DIR   = Path("/content/drive/MyDrive/gt-markets")
SRC_EXTRACT   = PROJECT_DIR/"app-demo"/"extracted"      # input model outputs
DST_ARTE      = PROJECT_DIR/"AppDemo"/"artefacts"       # final artefacts for app
DATA_PROCESSED= PROJECT_DIR/"data"/"processed"

ASSETS = ["GOLD","BTC","OIL","USDCNY"]
FREQ_DIRS = {"daily":"D","weekly":"W"}
ANNUALIZATION = {"D":252,"W":52}
COST_BPS = 5

# -----------------------
# Load engineered dataset (for Close prices)
# -----------------------
ENGINEERED_FILE = DATA_PROCESSED/"merged_financial_trends_engineered_2025-09-07.csv"
df_all = pd.read_csv(ENGINEERED_FILE, parse_dates=["Date"])

# Explicit ticker mapping
TICKER_MAP = {
    "GOLD":   ["GOLD", "GC=F"],
    "OIL":    ["OIL", "CL=F"],
    "BTC":    ["BTC", "BTC-USD"],
    "USDCNY": ["USDCNY", "USDCNY=X"],
}

# Build price map {asset -> DataFrame(Date, Close)}
PRICE_MAP = {}
for asset, tickers in TICKER_MAP.items():
    close_cols = [c for c in df_all.columns
                  if any(t.lower() in c.lower() for t in tickers) and "close" in c.lower()]
    if not close_cols:
        print(f"[warn] no Close column found for {asset}")
        continue
    col = close_cols[0]
    PRICE_MAP[asset] = (
        df_all[["Date", col]]
        .rename(columns={col:"Close"})
        .dropna()
        .reset_index(drop=True)
    )
    print(f"[info] mapped Close for {asset}: {col}")

# -----------------------
# Helpers
# -----------------------
def to_equity(close, pos, freq):
    """Convert positions into equity curve + metrics"""
    ret = close.pct_change().fillna(0)
    pos = pos.fillna(0).clip(0,1)
    strat = (pos.shift(1)*ret) - pos.diff().abs().fillna(0)*(COST_BPS/1e4)
    eq = (1+strat).cumprod()

    ann = ANNUALIZATION[freq]
    mu = strat.mean()*ann
    sd = strat.std()*math.sqrt(ann) if strat.std()>0 else np.nan
    sharpe = mu/sd if sd and sd>0 else np.nan
    maxdd = (eq/eq.cummax()-1).min()

    return eq, {"Return_Ann":mu, "Vol_Ann":sd, "Sharpe":sharpe, "MaxDD":maxdd}

def infer_asset(name):
    """Infer asset name from file path"""
    n = name.lower()
    if "gold" in n or "gc=f" in n: return "GOLD"
    if "btc" in n: return "BTC"
    if "oil" in n or "cl=f" in n: return "OIL"
    if "usdcny" in n: return "USDCNY"
    return None

def ensure_close(df, asset):
    """Attach Close column if missing"""
    if "Close" in df.columns and df["Close"].notna().any():
        return df
    if asset not in PRICE_MAP:
        print(f"[warn] No price data for {asset}")
        return df
    out = df.merge(PRICE_MAP[asset], on="Date", how="left", validate="m:1")
    miss = out["Close"].isna().sum()
    if miss:
        print(f"[warn] {asset}: {miss} rows still missing Close after merge")
    return out

# -----------------------
# Main loop
# -----------------------
for sub, F in FREQ_DIRS.items():
    src = SRC_EXTRACT/sub
    if not src.exists():
        continue
    for csv in src.rglob("*.csv"):
        asset = infer_asset(csv.name) or infer_asset(str(csv))
        if not asset:
            continue

        try:
            df = pd.read_csv(csv, parse_dates=["Date"])
        except:
            continue

        if "Date" not in df.columns:
            continue

        df = df.dropna(subset=["Date"])

        # --- fix Close column ---
        df = ensure_close(df, asset)
        if "Close" not in df.columns or df["Close"].isna().all():
            print(f"[skip] no Close for {csv.name}")
            continue

        # --- determine position ---
        if "position" in df.columns:
            pos = df["position"].astype(float)
        elif "prob_up" in df.columns:
            pos = (df["prob_up"]>0.5).astype(float)
        else:
            continue

        close = pd.to_numeric(df["Close"], errors="coerce")
        close.index = df["Date"]

        # --- strategy name (use model file stem) ---
        strat = "KW_" + csv.stem.replace(".","_").upper()

        out_dir = DST_ARTE/asset
        (out_dir/"figs").mkdir(parents=True, exist_ok=True)

        # --- write signals ---
        sig = pos.diff().fillna(0).replace({1.0:1, -1.0:0}).astype(int)
        pd.DataFrame({
            "Date": df["Date"],
            "signal": sig,
            "position": pos.astype(int),
            "Close": close.values,
            "strategy": strat
        }).to_csv(out_dir/f"signals_{strat}_{F}.csv", index=False)

        # --- metrics + fig ---
        eq, mets = to_equity(close, pos, F)
        ax = eq.plot(figsize=(6,3), title=f"{asset}-{strat}-{F}")
        ax.grid(True, alpha=0.3)
        ax.get_figure().savefig(out_dir/"figs"/f"{strat}_{F}.png", dpi=150, bbox_inches="tight")
        plt.close(ax.get_figure())

        row = {"asset":asset,"freq":F,"strategy":strat, **mets}
        out_csv = out_dir/f"metrics_keywords_{F}.csv"
        if out_csv.exists():
            old = pd.read_csv(out_csv)
            pd.concat([old, pd.DataFrame([row])]).drop_duplicates().to_csv(out_csv, index=False)
        else:
            pd.DataFrame([row]).to_csv(out_csv, index=False)

print("✅ Keyword artefacts rebuilt under", DST_ARTE)

Mounted at /content/drive
[info] mapped Close for GOLD: GC=F Close
[info] mapped Close for OIL: CL=F Close
[info] mapped Close for BTC: BTC-USD Close
[info] mapped Close for USDCNY: USDCNY=X Close
✅ Keyword artefacts rebuilt under /content/drive/MyDrive/gt-markets/AppDemo/artefacts
